# Gradient descent for Tilt and Thickness Optimization

Chia-Hao Lee, last update: 2025.09.03

## Dependencies
- abtem
- ase
- cupy
- torch
- torchvision
- matplotlib

## Major steps
1. Use abTEM as an initialization step, take the user-provided params (i.e., max thickness, unit cell) to generate potential and scan position (Ang)
2. Preprocess the potential into object phase shift, and scan position into pixels
3. Prepare the probe and multislice propagator (Need to be careful about the probe size at the end of the slice)
4. Initialize the AD model with optimizers and params. Probe and position are fixed, and prepare the object with a differentiable soft mask
5. Feed the scan position, object, probe, propagator into the multislice forward function to get diffraction pattern
6. Make PACBED out of all diffraction patterns 
7. Calculate the loss (We might be able to add reduction methods here)
8. Backpropagate the grad
9. Take the optimizer step to update the params (thickness, tilt_x, tilt_y) for 1 update step

## Notes
- Use broadcasting to process tilts and thicknesses in parallel
- Have an option ('exit_planes') to output all the CBED from intermediate thickness
  - This option would probably be used for qBO, not the GD baseline

In [ ]:
import os
import matplotlib.pyplot as plt

work_dir = "H:/workspace/bott"
os.chdir(work_dir)
print("Current working dir: ", os.getcwd())

## Configure abTEM parameter for the ground truth

In [ ]:
params_abtem = {
    # Input unknown
    'thickness': 1000, # Ang
    'tilt_x': 0, # mrad
    'tilt_y': 0, # mrad

    # Read cif into abTEM for 4D-STEM generation
    'path_crystal': './data/SrTiO3.cif',

    # Potential params
    'potential_extent_x': 62.6/2, #,62.6, # Ang
    'potential_extent_y': 62.6/2, #,62.6, # Ang
    'lateral_sampling': 0.2*2/3, # Ptycho recon pixel size = 0.2 Ang, so ptycho CBED need 1/2dx = 2.5 Ang-1 kMax. Considering the 2/3 kMax antialias, we need 2.5*1.5 = 3.75 Ang-1 for abTEM CBED or equivalently 0.1333 Ang px
    'vertical_sampling': 2,
    'potential_parametrization': "lobato", # Seems to be more accurate then "kirkland"
    'potential_projection': "finite", # infinite is faster but less accurate
    'exit_planes': 5, # Output diffraction pattern every N slices. Set to None to retun only the final diffraction pattern.

    # Phonon params
    'random_seed': 42,
    'use_frozen_phonon': False,
    'num_phonon_configs': 5,
    'phonon_sigma': {'Sr':.088,'Ti':.0746,'O':.0963}, #0.1 # Ang 
    
    # Probe params
    'energy': 200e3,
    'convergence_angle': 19.1,
    'df': 0,
    'aberrations': {},
    
    # Scan and pacbed
    'collection_angle': 'valid', # float or {'cutoff', 'valid', 'full'}. float angle would be in mrad. full means the diffraction pattern has the same size with potential, while 'cutoff' would crop to 2/3 kMax, and 'valid' returns only the inscribed squaure inside 'cutoff'.
    'scan_step_size': 0.8, # 0.3 to 0.8 Ang
    'return_pacbed': False
}

In [ ]:
from bott.gradient import SimuAbTEM

simu_abtem = SimuAbTEM(params_abtem, device='gpu', verbose=True)
measurement_arr = simu_abtem.simulate_cbed()

## Quick visualization of abTEM-simulated CBEDs

In [ ]:
# Reduce the measurement just for visualization purpose
if measurement_arr.ndim == 3:
    meas = measurement_arr.mean(0) # In case return_pacbed = False
else:
    meas = measurement_arr
print(f"meas.shape = {meas.shape}") # Note that I don't reduce the meas dimension because abTEM could flexibly add different dimensions with params distributions (i.e., thickness)

if meas.ndim == 2:
    meas_disp = meas
elif meas.ndim == 4:
    meas_disp = meas.sum(1)[-1]

plt.figure()
plt.imshow(meas_disp.T**0.5) # Transpose so ky is vertical, kx is horizontal
plt.colorbar()
plt.show()

## (OPTIONAL) Show the trend of total intensity, BF, and DF vs. thickness

- `exis_planes` must be a valid integer to return multiple thicknesses

In [ ]:
from bott.utils import create_circular_mask

convergence_angle = params_abtem['convergence_angle']
cutoff_angle = simu_abtem.alpha_max_final # Final cutoff angle
mask = create_circular_mask(height=meas.shape[-2], width=meas.shape[-1], radius=convergence_angle/cutoff_angle/2)

fig, axs = plt.subplots(1,3, figsize=(12,4))
axs[0].imshow(mask)
axs[0].set_title("BF mask")
axs[1].imshow(meas_disp.T)
axs[1].set_title("CBED")
axs[2].imshow(mask*meas_disp.T)
axs[2].set_title("masked CBED")

plt.show()

In [ ]:
pacbeds = meas.mean(-3) # Get pacbeds by averaging the probe position dimension
BF = mask[None,]*pacbeds
DF = (~mask[None,])*pacbeds

fig, axs = plt.subplots(1,3, figsize=(12,4))
plt.suptitle(f"Cutoff = {params_abtem['collection_angle']}")
axs[0].plot(pacbeds.sum((-2,-1)))
axs[0].set_title("Total intensity")
axs[0].set_xlabel('Thickness (nm)')

axs[1].plot(BF.sum((-2,-1)))
axs[1].set_title("BF intensity")
axs[1].set_xlabel('Thickness (nm)')

axs[2].plot(DF.sum((-2,-1)))
axs[2].set_title("DF intensity")
axs[2].set_xlabel('Thickness (nm)')
plt.show()

In [ ]:
fig, axs = plt.subplots(1,11, figsize=(35,3))
for i in range(11):
    axs[i].set_title(f"{10*i} nm")
    axs[i].set_xticks([])
    axs[i].set_yticks([])
    im = axs[i].imshow(pacbeds[10*i], vmin=0)
    fig.colorbar(im, shrink=0.5)
plt.show()

# GD baseline initialization

### TODO - 2025.08.14
- Maybe allow potential to be resampled along depth so we can reduce the forward pass time
- Maybe add a flag to enable sequential PACBED simulation to prevent OOM

In [ ]:
# This is duplicated from the previous section just for ease of use.
# While the params are different for computation efficiency

params_abtem = {
    # Input unknown
    'thickness': 100, # Ang
    'tilt_x': 20, # mrad
    'tilt_y': -20, # mrad

    # Read cif into abTEM for 4D-STEM generation
    'path_crystal': './data/SrTiO3.cif',

    # Potential params
    'potential_extent_x': 62.6, #,62.6, # Ang
    'potential_extent_y': 62.6, #,62.6, # Ang
    'lateral_sampling': 0.2*2/3, # Ptycho recon pixel size = 0.2 Ang, so ptycho CBED need 1/2dx = 2.5 Ang-1 kMax. Considering the 2/3 kMax antialias, we need 2.5*1.5 = 3.75 Ang-1 for abTEM CBED or equivalently 0.1333 Ang px
    'vertical_sampling': 2,
    'potential_parametrization': "lobato", # Seems to be more accurate then "kirkland"
    'potential_projection': "finite", # infinite is faster but less accurate
    'exit_planes': None, # Output diffraction pattern every N slices. Set to None to retun only the final diffraction pattern.

    # Phonon params
    'random_seed': 42,
    'use_frozen_phonon': False,
    'num_phonon_configs': 5,
    'phonon_sigma': {'Sr':.088,'Ti':.0746,'O':.0963}, #0.1 # Ang 
    
    # Probe params
    'energy': 200e3,
    'convergence_angle': 19.1,
    'df': 0,
    'aberrations': {},
    
    # Scan and pacbed
    'collection_angle': 'cutoff', # float or {'cutoff', 'valid', 'full'}. float angle would be in mrad. full means the diffraction pattern has the same size with potential, while 'cutoff' would crop to 2/3 kMax, and 'valid' returns only the inscribed squaure inside 'cutoff'.
    'scan_step_size': 0.3, # 0.3 to 0.8 Ang
    'return_pacbed': True
}

from bott.gradient import SimuAbTEM

simu_abtem = SimuAbTEM(params_abtem, device='gpu', verbose=True)
measurement_arr = simu_abtem.simulate_cbed()
print(f"measurement_arr.shape = {measurement_arr.shape}")

In [ ]:
import torch
import numpy as np
from bott.gradient import plot_cbed_difference, create_optimizer, GDTT, toggle_grad_requires

# Parse params from params_abtem. In practice, these params are unknown and need to be specified by users
lateral_sampling = params_abtem['lateral_sampling']
energy = params_abtem['energy']
convergence_angle = params_abtem['convergence_angle']
df = params_abtem['df']
vertical_sampling = params_abtem['vertical_sampling']

# These entries would need to be simulated by user via abTEM
pos_ang_yx = simu_abtem.pos_ang_yx
potential_arr = simu_abtem.potential_arr

# Setup the optimization parameters
Npix = 128 # This is the size of the CBED
dx = lateral_sampling * 3/2 # Ang, Note that this dx is assigned for the case of ground truth being cropped as "cutoff" antialias angle

# Note that the order and tilt_y, tilt_x and the directions are both reversed from abTEM to my own convention.
# I go with tilt_y, tilt_x, and my positive direction corresponds to abTEM's negative direction
# tilts and thicknesses can have multiple initial values for parallel optimization
init_tilts = np.array([[0,0], [5, -5], [15, -15]])
init_thicknesses = np.array([10, 50, 90])

# Initialize probe
probe_params = {
    "kv": energy/1e3,         # Ang
    "conv_angle": convergence_angle, # mrad
    "Npix": Npix,       # Number of pixel of thr detector/probe
    "dx": dx, # px size in Angstrom
    ## Aberration coefficients
    "df": df, #first-order aberration (defocus) in angstrom
    "c3": 0, #third-order spherical aberration in angstrom
    "c5": 0, #fifth-order spherical aberration in angstrom
    "c7":0, #seventh-order spherical aberration in angstrom
    "f_a2":0, #twofold astigmatism in angstrom
    "f_a3":0, #threefold astigmatism in angstrom
    "f_c3":0, #coma in angstrom
    "theta_a2":0, #azimuthal orientation in radian
    "theta_a3":0, #azimuthal orientation in radian
    "theta_c3":0, #azimuthal orientation in radian
    "shifts":[0,0], #shift probe center in angstrom
}

model_params = {
    'init_tilts': init_tilts, # mrad
    'init_thicknesses': init_thicknesses, # ang
    'dx': dx,
    'dz': vertical_sampling,
    'init_potential': potential_arr,
    'init_pos_ang_yx': pos_ang_yx,
    'raw_meas': measurement_arr, # This is currently provided by abTEM simulation. In practice, this would be provided by user.
    'probe_params': probe_params,
    'scale_dp': 1e4,
    'scale_tilt': 10, # Set the scale to "maximum expected tilt in mrad" so we're optimizing between 0-1 
    'scale_thickness': 100, # Set the scale to "maximum expected thickness in Ang" so we're optimizing between 0-1 
    'return_pacbed': True, # This will average across scan positions
    'meas_flipT': [0,0,1], # For abTEM simulation, an x-y transpose is needed
    'detector_blur_std'   : None, # type: None or float, unit: px (k-space). This applies Gaussian blur to the forward model simulated diffraction patterns to emulate the PSF of high-energy electrons on detector for experimental data. Typical value is 0-1 px (std) based on the acceleration voltage 
    'optimizer_params'    : {'name': 'Adam', 'configs': {}}, # Support all PyTorch optimizer except LBFGS because LBFGS can't set separate learning rates for different tensors. The suggested optimizer is 'Adam' with default configs (None). You can load the previous optimizer state by passing the path of `model.hdf5` to `load_state`, this way you can continue previous reconstruciton smoothly without abrupt gradients. (Because lots of the optimizers are adaptive and have history-dependent learning rate manipulation, so loading the optimizer state is necessary if you want to continue the previous optimization trajectory). However, the optimizer state must be coming from previous reconstructions with the same set of optimization variables with identical size of the dimensions otherwise it won't run.
    'update_params':{
        'tilts'       : {'start_iter': 1, 'lr': 1e-2, 'end_iter': None}, # object tilts
        'thicknesses' : {'start_iter': 200, 'lr': 1e-2, 'end_iter': None}, # object dz slice thickness 
    }
}

indices = np.random.choice(np.arange(196), size=1, replace=False)

model   = GDTT(model_params, device='cuda', verbose=True)
dp_fwd  = model(indices) # (indices, num_tilts, num_thicknesses, ky, kx)
dp_meas = model.get_dp_meas() # (ky, kx)

for i in range(len(init_tilts)):
    for j in range(len(init_thicknesses)):
        print(f"")
        print(f"tilt: {init_tilts[i]} mrad")
        print(f"thicknesses: {init_thicknesses[j]} Ang")
        plot_cbed_difference(dp_fwd[i,j].detach().cpu().numpy(), dp_meas.squeeze().detach().cpu().numpy())


# GD Optimization loop

In [ ]:
# recon_params
NITER = 500
dp_pow = 0.5
batch_size = 1 # 'batch_size' determines how many probe positions are used to generate the PACBED. Set to 1 for efficiency but it tends to underestimate thickness

# Initialization
model = GDTT(model_params, device='cuda', verbose=True)
loss_fn = torch.nn.MSELoss(reduction='none')
optimizer = create_optimizer(model.optimizer_params, model.optimizable_params)

# Optimization loop
for niter in range(1,NITER+1):
    
    # Toggle the grad calculation to enable or disable AD update on tensors at certain iterations
    toggle_grad_requires(model, niter, verbose=False)
    
    indices = np.random.choice(np.arange(196), size=batch_size, replace=False) 
    
    # Get loss, backward, and update
    optimizer.zero_grad()
    dp_fwd  = model(indices)
    dp_meas = model.get_dp_meas().broadcast_to(dp_fwd.shape)
    loss = loss_fn(dp_fwd**dp_pow, dp_meas**dp_pow)
    loss = loss.mean(dim=(-2,-1)) # Average the loss across spatial dimension so it's (num_tilt, num_thickness), i.e, N-Dimensional loss
    loss.backward(torch.ones_like(loss)) # This allows each individual case to have their own loss
    optimizer.step() # Note that during update, the losses with shared params (i.e., same tilts) would be summed together so it's not 100% decoupled
    
    # Convert tensors for display/record keeping
    loss_np = loss.detach().cpu().numpy()
    tilts_np = model.get_tilts().detach().cpu().numpy()
    thicknesses_np = model.get_thicknesses().detach().cpu().numpy()
    
    # Record keeping
    model.loss_iters.append(loss_np)
    model.tilts_iters.append(tilts_np)
    model.thicknesses_iters.append(thicknesses_np)
    
    # Logging
    if niter % 10 == 0:
        print(f"")
        print(f"Iter: {niter}, loss: {loss_np}")
        print(f"tilts: {tilts_np}")
        print(f"thicknesses: {thicknesses_np}")

In [ ]:
tilts_iters = np.array(model.tilts_iters)
thicknesses_iters = np.array(model.thicknesses_iters)
loss_iters = np.array(model.loss_iters)
niter = len(loss_iters)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Automatically detect dimensions and create subplots
loss_iters = np.array(model.loss_iters)  # Ensure loss_iters is a NumPy array
niter = len(loss_iters)

# Get the shape of the last two dimensions
num_tilts, num_thicknesses = loss_iters.shape[1], loss_iters.shape[2]

# Create subplots
fig, axs = plt.subplots(num_tilts, num_thicknesses, figsize=(15, 10), squeeze=False)
fig.suptitle("Loss Iterations", fontsize=16)

for i in range(num_tilts):
    for j in range(num_thicknesses):
        axs[i, j].plot(loss_iters[:, i, j])
        axs[i, j].set_title(f"Tilt {init_tilts[i]} mrad \nThickness {init_thicknesses[j]} Ang")
        axs[i, j].set_xlabel("Iteration")
        axs[i, j].set_ylabel("Loss")

# Adjust layout
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 3, figsize=(15, 5))  # Create a 1x3 subplot

# Plot tilt y
axs[0].plot(tilts_iters[:, :, 0])
axs[0].set_title('Tilt Y')
axs[0].set_xlabel('Iteration')
axs[0].set_ylabel('Tilt Y (mrad)')

# Plot tilt x
axs[1].plot(tilts_iters[:, :, 1])
axs[1].set_title('Tilt X')
axs[1].set_xlabel('Iteration')
axs[1].set_ylabel('Tilt X (mrad)')

# Plot thickness
axs[2].plot(thicknesses_iters[:, :])
axs[2].set_title('Thickness')
axs[2].set_xlabel('Iteration')
axs[2].set_ylabel('Thickness (Ang)')

# Adjust layout
plt.tight_layout()
plt.show()